# Requires having blast installed.

In [7]:
import os
import subprocess

fasta_parts = [
    "data/training_data/animal_training_part1.fasta",
    "data/training_data/animal_training_part2.fasta",
    "data/training_data/animal_training_part3.fasta"
]

output_path = "data/aling_db/animal_training.fasta"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w") as outfile:
    for file in fasta_parts:
        with open(file, "r") as infile:
            outfile.write(infile.read().rstrip() + "\n")

## Specify the path to the makeblastdb.exe file of the blast tool.

In [9]:
makeblastdb_path = r"C:\Users\Bruno\Downloads\ncbi-blast-2.16.0+\bin\makeblastdb.exe"

In [10]:
cmd = [
    makeblastdb_path,
    "-in", output_path,
    "-dbtype", "prot" 
]

try:
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    print("Database created successfully!")
    print(result.stdout)
except subprocess.CalledProcessError as e:
    print("Error creating the database:")
    print(e.stderr)

Database created successfully!


Building a new DB, current time: 02/15/2026 10:40:06
New DB name:   C:\Users\Bruno\Downloads\Bielefeld_project\git\data\aling_db\animal_training.fasta
New DB title:  data/aling_db/animal_training.fasta
Sequence type: Protein
Deleted existing Protein BLAST database named C:\Users\Bruno\Downloads\Bielefeld_project\git\data\aling_db\animal_training.fasta
Keep MBits: T
Maximum file size: 3000000000B
Adding sequences from FASTA; added 661545 sequences in 51.4108 seconds.





## Specify the path to the fasta file you want to align with the training and the output name.

In [12]:
query_fasta = f"data/test_data/arquea_test.fasta" 
output_file = f"data/test_data/arquea_test_aling.txt"

## Specify the path to blastp.exe

In [14]:
blastp_path = r"C:\Users\Bruno\Downloads\ncbi-blast-2.16.0+\bin\blastp.exe" 

In [ ]:
import subprocess

db_path = output_path

cmd = [
    blastp_path,
    "-query", query_fasta,
    "-db", db_path,
    "-out", output_file,
    "-outfmt", "6 qseqid sseqid pident ppos nident mismatch gapopen qlen slen length qstart qend sstart send evalue bitscore",
    "-max_target_seqs", "5",
    "-num_threads", "40"
]

try:
    result = subprocess.run(cmd, capture_output=True, text=True, check=True)
    print("BLAST completed successfully!")
    print(f"Resultados salvos em: {output_file}")
except subprocess.CalledProcessError as e:
    print("Error running BLAST:")
    with open("error_aling.txt", "w") as f:
      f.write(e.stderr)
    print(e.stderr)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

colunas = [
    "qseqid", "sseqid", "pident", "ppos", "nident", "mismatch", "gapopen",
    "qlen", "slen", "length", "qstart", "qend", "sstart", "send", "evalue", "bitscore"
]
df2 = pd.read_csv(
    output_file, 
    sep="\t",
    header=None,
    names=colunas
)

df2['fraction'] = (-df2['qstart']+1+df2['qend'])/df2['qlen']
df2['score'] = (df2['pident'] * df2['fraction']).round(2)
df_max = df2.loc[df2.groupby("qseqid")["score"].idxmax()]
df_max = df_max.iloc[1:]
df_max = df_max[['qseqid','pident','qstart', 'qend', 'qlen', 'fraction', 'score']]
df_max.to_csv(output_file.replace(".txt", ".csv"), index=False)
plt.hist(df_max["score"], bins=50, edgecolor='black')
print('Your alignment has been completed and is in:', output_file.replace(".txt", ".csv"))